In [31]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report

In [32]:
data=pd.read_csv('insurance.csv')


In [33]:
def incurence_premimu_category(ex,sm,age):
    if sm=='no':
        return 'low'
    elif ex>=12000 and sm=='yes' and age<30:
        return 'Medium'
    elif ex>=12000 and sm=='yes' and age>30:
        return 'heigh'
    else:
        return 'low'
        

In [34]:
data['incurence_premium']=data.apply(lambda row:incurence_premimu_category(row['expenses'],row['smoker'],row['age']),axis=1)

In [35]:
data['incurence_premium'].value_counts()
#data=data.drop(columns='incurence_premium_category')

incurence_premium
low       1073
heigh      179
Medium      86
Name: count, dtype: int64

In [36]:
data.region.unique()

array(['southwest', 'southeast', 'northwest', 'northeast'], dtype=object)

In [37]:
x=data[['age','sex','bmi','smoker','region','expenses']]
y=data['incurence_premium']
#x = x.replace([None], np.nan)

In [38]:
categorial_features=['sex','smoker','region']
numerical_fearure=['bmi','age','expenses']

In [39]:
preprocessor=ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(),categorial_features),
        ("num",'passthrough',numerical_fearure)
    ]
)


In [40]:
pipline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('classifier',RandomForestClassifier(random_state=42))
])

In [41]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
pipline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [42]:
y_pres=pipline.predict(X_test)
accuracy_score(y_test,y_pres)

print(classification_report(y_test,y_pres))

              precision    recall  f1-score   support

      Medium       1.00      1.00      1.00        12
       heigh       1.00      1.00      1.00        40
         low       1.00      1.00      1.00       216

    accuracy                           1.00       268
   macro avg       1.00      1.00      1.00       268
weighted avg       1.00      1.00      1.00       268



In [43]:
import pickle

pickle_model_path='model.pkl'
with open(pickle_model_path,'wb') as f:
    pickle.dump(pipline,f)